In [0]:
import requests
import base64
import uuid
import pandas as pd
from pyspark.sql.functions import current_timestamp, lit
from datetime import datetime

# Configuration
API_BASE_URL = "https://api.onegov.nsw.gov.au"
OAUTH_ENDPOINT = "/oauth/client_credential/accesstoken"
FUEL_PRICES_ENDPOINT = "/FuelPriceCheck/v2/fuel/prices"
BASIC_AUTH = "Basic WXEwT21JYzZHT1NGZUJDTW5oU3BCWGFoTm1GaVlQOGo6a1RmVXBwc3lpMUJpUm5OOA=="

# Extract API Key from Basic Auth
auth_decoded = base64.b64decode(BASIC_AUTH.replace("Basic ", "")).decode('utf-8')
client_id, client_secret = auth_decoded.split(':')
API_KEY = client_id

# Step 1: Get OAuth Access Token
oauth_url = API_BASE_URL + OAUTH_ENDPOINT
oauth_response = requests.get(
    oauth_url,
    headers={"Authorization": BASIC_AUTH},
    params={"grant_type": "client_credentials"}
)

if oauth_response.status_code != 200:
    raise Exception(f"OAuth failed: {oauth_response.status_code} - {oauth_response.text}")

access_token = oauth_response.json().get('access_token')
if not access_token:
    raise Exception("No access token in OAuth response")

# Step 2: Fetch Fuel Prices
fuel_url = API_BASE_URL + FUEL_PRICES_ENDPOINT
transaction_id = str(uuid.uuid4())
request_timestamp = datetime.utcnow().strftime("%d/%m/%Y %I:%M:%S %p")

fuel_response = requests.get(
    fuel_url,
    headers={
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json; charset=utf-8",
        "apikey": API_KEY,
        "transactionid": transaction_id,
        "requesttimestamp": request_timestamp
    },
    params={"states": "NSW"}
)

if fuel_response.status_code != 200:
    raise Exception(f"API request failed: {fuel_response.status_code} - {fuel_response.text}")

data = fuel_response.json()

# Extract records
if isinstance(data, dict) and 'prices' in data:
    records = data['prices']
elif isinstance(data, list):
    records = data
else:
    for key, val in data.items():
        if isinstance(val, list) and len(val) > 0:
            records = val
            break
    else:
        raise Exception("Unable to find data array in response")

# Create DataFrame
pdf = pd.DataFrame(records)
df_raw = spark.createDataFrame(pdf)

# Add metadata
df_with_metadata = df_raw \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("source_system", lit("NSW_FuelCheck_API"))

record_count = df_with_metadata.count()

print(f"Fetched {record_count:,} fuel price records from NSW API")
display(df_with_metadata.limit(10))

In [0]:
# Write to Delta table
FULL_TABLE_NAME = "mobility_ai.bronze.fuel_raw"

df_with_metadata.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(FULL_TABLE_NAME)

print(f"Ingestion Complete: {record_count:,} records written to {FULL_TABLE_NAME}")